![Web Automation with Selenium](https://github.com/ValRCS/RTU_Automating_Tasks_With_Python/blob/main/img/Topic_7_Web_Automation_Selenium.png?raw=true)

# Topic 7 — Web Automation with Selenium

## 1. Goals of This Notebook
In this notebook you will learn:
- **Why Selenium exists** and when it is needed for scraping or automation.  
  Selenium controls a real browser and can render dynamic JavaScript-based content that normal `requests` cannot retrieve.
- **How to install and configure Selenium on Windows**, including notes on drivers, PATH issues, and running inside Jupyter.
- **How to create browser sessions**, navigate, extract data, wait for elements, click buttons, scroll pages, and automate tasks.
- **A full scraping example** using a safe demo website.

---



## 2. What is Selenium?

Selenium official site: https://www.selenium.dev/

Selenium is a **browser automation framework** that allows Python (and many other languages) to control a real web browser such as Chrome, Firefox, or Edge. Unlike traditional web scraping tools that download raw HTML, Selenium operates exactly like a human user—opening pages, clicking buttons, filling forms, executing JavaScript, and waiting for dynamic content to appear.

Selenium is essential when a website:

- Loads content **only after JavaScript executes**  
- Uses **infinite scroll**, pop-ups, modals, or interactive UI components  
- Requires **login sequences**, form submissions, or cookies that must persist between requests  
- Blocks automated HTTP requests or returns incomplete HTML  
- Behaves differently depending on the browser state, viewport size, or JavaScript interactions  

In short, **Selenium gives you full control of an actual browser**, making it suitable for both automated testing and scraping modern, dynamic websites that do not expose their data through static HTML or APIs.


## 2b. Comparison With Requests + BeautifulSoup (Video 6 Recap)

### When Requests + BS4 *is enough*:
- Static HTML pages  
- Simple extraction  
- No interactions  
- No JavaScript rendering  
- Very fast and lightweight  

### When Requests + BS4 *fails*:
- Page loads content **only after JavaScript runs**
- Websites with pagination that loads dynamically
- Login-required sites with CSRF tokens
- Infinite scrolling / lazy loading

### Why Selenium:
- Executes JavaScript  
- Can click, type, scroll, and interact like a real user  
- Good for:
  - dynamic sites  
  - testing workflows  
  - scraping where no clean API exists  

---

## 3. Installing Selenium on Windows

### Step 1 — Install Selenium Python package
Run in terminal or inside Jupyter:

```
pip install selenium
```

### Step 2 — Browser Driver (Modern Approach: Selenium Manager)
As of Selenium 4.10+, Selenium **automatically** downloads the correct browser driver for:
- Chrome  
- Edge  
- Firefox  

**No need to separately download chromedriver.exe.**

### Step 3 — Common Windows Pitfalls
- **PATH issues**: older Selenium tutorials require manual driver installation — ignore those.
- **Antivirus blocking**: some AV tools block automated browser control.  
- **Browser version mismatch**: Selenium Manager eliminates this problem.
- **Running inside Jupyter**:
  - Jupyter blocks some interactive window features  
  - Browser will launch in a new system window  

### Step 4 — Verify Installation
We’ll run a minimal script below.

---

## 4. Choosing a Practice Website

We will use:

### **Quotes to Scrape (JavaScript version)**  
https://quotes.toscrape.com/js

It requires JavaScript, so standard requests fail — perfect for Selenium demos.

---

## 5. Selenium Basics

### 5.1 Creating a Browser Instance
We use Chrome by default. Selenium will handle driver installation.

### 5.2 Opening a Page
We navigate using `.get(url)`.

### 5.3 Locating Elements
Use:
- `By.CSS_SELECTOR`  
- `By.XPATH`  
- `By.CLASS_NAME`  
- `By.TAG_NAME`

### 5.4 Extracting Content
Every Selenium element supports:
- `.text`  
- `.get_attribute("href")`  

### 5.5 Waiting for Dynamic Content
Use:  
`WebDriverWait(driver, timeout).until(condition)`

### 5.6 Interactions
Selenium supports:
- `.click()`  
- `.send_keys()`  
- `.execute_script()` for scrolling


---


## Selenium in 2025 — How Driver Management Works

As of Selenium 4.10+ (2023) and continuing through 2025, Selenium ships with a built-in tool called Selenium Manager.

### What Selenium Manager does

Detects which browser you’re using (Chrome, Edge, Firefox).

Automatically downloads the correct matching driver version.

Caches the driver locally.

Handles updates when your browser updates.

Eliminates manual driver installation completely.

### Old tutorials are outdated

What you do NOT need to install anymore:

❌ chromedriver.exe

❌ geckodriver.exe

❌ msedgedriver.exe

❌ Add anything to PATH

❌ Download binaries manually

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By # for selecting elements
# other imports can be added as needed
# from selenium.webdriver.chrome.service import Service # uncomment if needed in future
# from selenium.webdriver.support.ui import WebDriverWait # uncomment if needed in future
# from selenium.webdriver.support import expected_conditions as EC

# Minimal test script — open python.org
driver = webdriver.Chrome()  # Selenium Manager auto-installs driver

# we actually open a website here using our robot browser - Chrome in this example
# essentially we are making a get request to the server hosting python.org
driver.get("https://www.python.org")

# datetime
from datetime import datetime
print("Current date and time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Title:", driver.title)
# we close the browser window
driver.quit()


Current date and time: 2025-12-07 22:20:56
Title: Welcome to Python.org


In [ ]:
# how about we keep the driver running for now we will quite in another cell
driver = webdriver.Chrome()  # Selenium Manager auto-installs driver if needed
driver.get("https://www.python.org")
print("Title:", driver.title)
# print first 5 links on the page
links = driver.find_elements(By.TAG_NAME, "a") # so instead of a any valid HTML tag can be used
# reminder on HTML tags at MDN: https://developer.mozilla.org/en-US/docs/Web/HTML/Element/a
for link in links[:10]: # so by slicing we get first 10 links, adjust as needed
    print(link.get_attribute("href")) # so get_attribute gets the href attribute of the <a> tag in this case

Title: Welcome to Python.org
https://www.python.org/#content
https://www.python.org/#python-network
https://www.python.org/
https://www.python.org/psf/
https://docs.python.org/
https://pypi.org/
https://www.python.org/jobs/
https://www.python.org/community/
https://www.python.org/#top
https://www.python.org/


In [5]:
# let's quit the driver now - close the browser
driver.quit() # in a .py script this would be one of the last commands

---

## 6. Practical Selenium Demonstration: Scraping Quotes

We target **https://quotes.toscrape.com/js**  
This version loads quotes via JavaScript and requires Selenium.

Process:
1. Open page  
2. Wait for quotes to load  
3. Extract quote text + author  
4. Click “Next”  
5. Repeat for 3 pages  
6. Save results to CSV  

---


In [6]:
import csv # csv is from standard library we could have used pandas as well
# from selenium import webdriver # we have this already imported
# from selenium.webdriver.common.by import By # we have this already imported
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()

# so we go to specific ULR
url = "https://quotes.toscrape.com/js" # maybe you would read this url from some config or input
# again url is just a string
driver.get(url) # this opens the page in the browser

all_quotes = [] # this is our storage (list data structure) for quotes

for page in range(3):  # scrape first 3 pages
    print(f"Scraping page {page+1}")

    # Wait for quote elements - this is very important for JS rendered pages - page might take time to load!!!
    quotes = WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CLASS_NAME, "quote"))
    )

    for q in quotes:
        text = q.find_element(By.CLASS_NAME, "text").text
        author = q.find_element(By.CLASS_NAME, "author").text
        all_quotes.append([text, author])

    # Try clicking the next button if exists
    try:
        next_btn = driver.find_element(By.CSS_SELECTOR, "li.next > a")
        next_btn.click() # this actually clicks that element!
    except:
        # exception occurs if no next button found
        print("No next page.")
        break

driver.quit()

# at this moment we have 0 or more quotes in all_quotes list

# Save results
with open("quotes_output.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["quote", "author"])
    writer.writerows(all_quotes) # if no quotes this writes nothing after header

all_quotes[:5]  # show sample


Scraping page 1
Scraping page 2
Scraping page 3


[['“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”',
  'Albert Einstein'],
 ['“It is our choices, Harry, that show what we truly are, far more than our abilities.”',
  'J.K. Rowling'],
 ['“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”',
  'Albert Einstein'],
 ['“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”',
  'Jane Austen'],
 ["“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”",
  'Marilyn Monroe']]

---

## 7. Scrolling & Screenshot Example

Useful for:
- Infinite scroll pages
- Triggering lazy-loaded content
- Capturing evidence of state

---


In [7]:
# from selenium import webdriver # if not already imported
import time

driver = webdriver.Chrome()
driver.get("https://quotes.toscrape.com/js")

# Scroll to bottom
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(2) # this sleeps for 2 seconds to allow any lazy loading to complete - adjust as needed


driver.save_screenshot("selenium_screenshot.png")
# we could have also obtained content as before by quotes = driver.find_elements(By.CLASS_NAME, "quote") etc

# also think of the idea of using image processing from previous lessons and getting some information from the screenshot!

driver.quit()


---

## 8. Cleanup, Tips & Future Work

- Selenium is slower than Requests/BS4 — use only when necessary.  
- Combine both approaches:
  - Selenium to log in / load dynamic HTML  
  - Requests to fetch data behind authenticated session  
- For production, run Selenium in:
  - **Headless mode**
  - **Docker containers**
  - **Selenium Grid** for parallel browsing  

End of Notebook.


---

# 9. Additional Examples for Extended Session 

This section expands Selenium practical examples for classroom use.


## 9.1 Login Automation

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()
driver.get("https://the-internet.herokuapp.com/login")

WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "username")))

# good practice would be to get the keys and password from some config or environment or secret manager
# for now I will emulate that with hardcoded strings
username = "tomsmith"
password = "SuperSecretPassword!"
# TODO in real life DO NOT hardcode credentials in code!!!!

driver.find_element(By.ID, "username").send_keys(username)
driver.find_element(By.ID, "password").send_keys(password)
# so we will click a button which has class radius
# find_element returns FIRST matching element - so if there were multiple buttons with that class only first is clicked

driver.find_element(By.CSS_SELECTOR, "button.radius").click() # it could fail if the page is slow to load or if there is no such element
# read up on your CSS selectors if needed: https://developer.mozilla.org/en-US/docs/Web/CSS/CSS_Selectors

msg = WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.ID, "flash"))).text
print(msg)

driver.quit()

You logged into a secure area!
×


## 9.2 File Download Example

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import os

# download_dir = os.path.abspath("downloads")
# download dir will be current working directory
download_dir = os.path.join(os.getcwd(), "downloads")
if not os.path.exists(download_dir):
    os.makedirs(download_dir)

opts = Options()
opts.add_experimental_option("prefs", {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False
})

driver = webdriver.Chrome(options=opts)
driver.get("https://the-internet.herokuapp.com/download")

# wait for the page to load for specific element here : a[href='download/some-file.txt']"
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "a[href='download/some-file.txt']"))
)
# again adjust as needed for different files

file_link = driver.find_element(By.CSS_SELECTOR, "a[href='download/some-file.txt']")
file_link.click()

print("Download started")

# maybe we should have waited until the file was downloaded completely
# just a simple delay would have been sufficient for demo purposes
time.sleep(5)  # adjust as needed for larger files or slower connections
# otherwise you might get a download but it will not have the correct name
driver.quit()

Download started


## 9.3 Infinite Scroll Demo

In [ ]:
from selenium import webdriver
import time

driver = webdriver.Chrome()
driver.get("https://the-internet.herokuapp.com/infinite_scroll")

for _ in range(10):
    # exectue_script actually runs JavaScript code in the context of the page - VERY POWERFUL!!!
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)

driver.quit()

## 9.4 Headless Mode

What is headless mode?

Headless mode allows you to run a web browser without a graphical user interface (GUI). In this mode, the browser operates in the background, performing all actions as if it were running normally, but without displaying any windows or visual elements on the screen. This is particularly useful for automated tasks, such as web scraping or testing, where you don't need to see the browser's interface.

In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

opts = Options()
opts.add_argument("--headless")
opts.add_argument("--window-size=1920,1080") # so we pretend we have a HD screen and off we go

driver = webdriver.Chrome(options=opts)
driver.get("https://www.python.org")
print(driver.title)
driver.quit()

## 9.5 Dropdown Example

In [13]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select

driver = webdriver.Chrome()
driver.get("https://the-internet.herokuapp.com/dropdown")

# again you can insert a wait here if needed

# so if an element has an ID that is a sure way to select it
# again we could have used CSS_SELECTOR or XPATH if needed
dropdown = Select(driver.find_element(By.ID, "dropdown"))
dropdown.select_by_visible_text("Option 2")

driver.quit()

## 9.6 Iframe Handling

iframes are HTML documents embedded within another HTML document. They are commonly used to display content from another source, such as advertisements, videos, or external web pages, within a webpage. When working with Selenium, interacting with elements inside an iframe requires switching the driver's context to the iframe first.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
driver.get("https://the-internet.herokuapp.com/iframe")

# wait for mce_0_ifr element to be available
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.ID, "mce_0_ifr"))
)


driver.switch_to.frame(driver.find_element(By.ID, "mce_0_ifr"))
editable = driver.find_element(By.ID, "tinymce")
# print if we found it
print("Found editable element inside iframe:", editable is not None)

# the following commands would work if we wanted to clear and send keys 
# editable.clear()
# editable.send_keys("Hello inside iframe!")

driver.switch_to.default_content()
driver.quit()

Found editable element inside iframe: True


## 9.7 Automating a Search Form

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Chrome()
driver.get("https://duckduckgo.com")

# wait 10 seconds - adjust as needed
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.TAG_NAME, "input"))
)

# find all input elements
inputs = driver.find_elements(By.TAG_NAME, "input")
# find inputs that are not hidden
visible_inputs = [i for i in inputs if i.is_displayed()]
# check if there are any visible inputs
if not visible_inputs:
    raise Exception("No visible input elements found")
    # we stop here then
    # exit() we could add an exit in a script if needed
# search will be first visible input
search = visible_inputs[0] # first input element that is visible

# search = driver.find_element(By.ID, "search_form_input_homepage")
search.send_keys("Selenium Python", Keys.RETURN)

results = WebDriverWait(driver, 10).until(
    EC.visibility_of_all_elements_located((By.CSS_SELECTOR, "a.result__a"))
)

print([r.text for r in results[:5]])
# TODO get actual links to these results!
print([r.get_attribute("href") for r in results[:5]])
# of course DuckDuckGo might change their page structure so selectors might need to be updated
# also pretty sure DuckDuckGo has some API for searching that would be better to use in real life

driver.quit()

['selenium python read the docs', 'selenium python documentation', 'selenium python edge', 'selenium python download', 'pip install selenium']


## 9.8 Extracting a Dynamic Table

In [16]:
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
driver.get("https://the-internet.herokuapp.com/tables")

rows = driver.find_elements(By.CSS_SELECTOR, "#table1 tr")

for row in rows:
    cols = [c.text for c in row.find_elements(By.TAG_NAME, "td")]
    if cols:
        print(cols)

# TODO exercise idea save in a pandas dataframe - students try it!

driver.quit()

['Smith', 'John', 'jsmith@gmail.com', '$50.00', 'http://www.jsmith.com', 'edit delete']
['Bach', 'Frank', 'fbach@yahoo.com', '$51.00', 'http://www.frank.com', 'edit delete']
['Doe', 'Jason', 'jdoe@hotmail.com', '$100.00', 'http://www.jdoe.com', 'edit delete']
['Conway', 'Tim', 'tconway@earthlink.net', '$50.00', 'http://www.timconway.com', 'edit delete']


## What we've done

In this session, we have:

- Compared **Requests + BeautifulSoup** with **Selenium**, and clarified when browser automation is required.
- Installed and configured **Selenium on Windows**, including how Selenium Manager now handles browser drivers automatically.
- Explored essential Selenium capabilities:
  - Launching browsers (Chrome/Edge/Firefox)
  - Navigating pages
  - Locating elements (CSS selectors, XPath)
  - Extracting text and attributes
  - Clicking buttons, typing into fields, and submitting forms
  - Waiting for dynamic content using `WebDriverWait`
- Built a full scraping workflow using the **JavaScript version** of *Quotes to Scrape*.
- Added practical automation examples:
  - Login automation
  - File downloads
  - Infinite scroll handling
  - Headless mode execution
  - Dropdown selection
  - Working with iframes
  - Automating searches
  - Extracting dynamic tables

These examples together form a complete introduction to modern browser automation and dynamic web scraping with Selenium.


## Exercise Ideas

### 1. Scrape **ss.com** With Selenium (Compare With BeautifulSoup Approach)

Try revisiting the earlier exercise where we used **Requests + BeautifulSoup** to scrape listings from *ss.com*.  
Now attempt the same task using **Selenium**:

**Tasks:**
- Open the main classifieds page (e.g., Cars → BMW).
- Wait for listings to load.
- Extract:
  - Title
  - Price
  - Year / mileage (if available)
  - Listing URL
- Scroll to load additional results (if any).
- Compare results with the BeautifulSoup version:
  - **What remains the same?** (HTML structure, general parsing logic)
  - **What is different?** (dynamic elements, pagination behavior, delays, cookies, JavaScript rendering)
- Discuss whether Selenium is actually required for ss.com or whether Requests was sufficient.

This exercise helps students identify when Selenium is necessary and when it is overkill.


### 2. Scrape a Dynamic Site That **BeautifulSoup Cannot Handle**

Choose a site where content loads **only after JavaScript execution**, making Requests/BS4 ineffective.  
A good teaching example is:

**https://quotes.toscrape.com/js/**  
(the JavaScript version)

**Tasks:**
- Load the page with Selenium.
- Wait for `.quote` elements to appear.
- Extract:
  - Quote text
  - Author name
- Navigate to the next page using the “Next →” button.
- Collect quotes from at least 3 pages.
- Save results into CSV.

**Goal:**  
Clearly demonstrate *why* BS4 fails here (static HTML is empty) and how Selenium handles dynamic rendering.


### 3. (Optional Advanced Idea) Infinite Scroll Scraper

Choose a site with infinite scroll such as:

**https://the-internet.herokuapp.com/infinite_scroll** 

**Tasks:**
- Scroll down repeatedly with `execute_script`.
- Detect when new content loads.
- Extract all loaded items into a list.

This example shows how Selenium handles modern UI patterns that Requests/BS4 cannot touch.



## References and Tutorials

### References

- **Selenium Official Documentation**  
  https://www.selenium.dev/documentation/  
  The authoritative source for Selenium 4+, covering WebDriver APIs, Selenium Manager (automatic driver handling), waits, actions, and best practices.

- **Selenium WebDriver API (Python Bindings)**  
  https://www.selenium.dev/selenium/docs/api/py/  
  Reference for all Python classes and methods, including WebDriver, WebElement, By, Expected Conditions, Select, and more.

- **Chromium WebDriver (W3C Standard)**  
  https://w3c.github.io/webdriver/  
  The underlying specification Selenium 4 implements. Useful for understanding the behavior of standardized browser automation.

- **Selenium GitHub Repository**  
  https://github.com/SeleniumHQ/selenium  
  Source code, issue tracker, release notes, and ongoing development. Helpful for checking changes to Selenium Manager, headless modes, and browser compatibility.


### Tutorials

- **Selenium with Python (selenium.dev guide)**  
  https://www.selenium.dev/documentation/webdriver/getting_started/  
  Official beginner-friendly walkthrough for writing Selenium scripts in Python. Includes examples for locating elements, waits, interactions, and navigating pages.

- **Real Python – Modern Web Scraping With Selenium**  
  https://realpython.com/modern-web-automation-with-python-and-selenium/
  Practical end-to-end examples including screenshots, waits, dynamic content, and practical scraping patterns.

- **Browser Automation with Selenium (TestAutomationU)**  
  https://testautomationu.applitools.com/selenium-webdriver-python-tutorial/  
  Free course with short video modules that introduce Selenium automation fundamentals using Python.

- **Headless Chrome Automation Guide**  
  https://developers.google.com/web/updates/2017/04/headless-chrome  
  Google’s official explanation of headless Chrome, command-line flags, and browser behavior in automated environments. Still relevant for Selenium 4+.
  There might be an update on this page in the future.

